## Chronological Split on Day Boundaries (all horizons)

**What changed vs. notebook 03:**

1. **Split at midnight, not at a row index.** Your original split cut the data at 05:00 (train/val)
   and 01:15 (val/test). Evaluating daytime performance then means partial days at the edges.
   Splitting by calendar day gives whole days in each set, so every day-level summary is clean.
2. **One split for all horizons.** The same calendar days go to train/val/test for h=1, 2, 4, so
   results are comparable across horizons.
3. **Boundary purge.** A row is kept only if its *target time* falls in the same split as its
   feature time. Without this, the last training rows would have targets inside the validation period.
   (At night the power is zero so the effect is tiny, but this makes the no-leakage claim exact.)
4. **Weather character of each split.** Daily insolation is printed so you can describe whether the
   4–5 day test window was sunny or cloudy.

Lag features of a validation/test row may reach back into the previous split. That is fine: those
are past values that would be available at forecast time.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import pandas as pd
import numpy as np

folder = "/content/drive/MyDrive/Project2_SolarPV/"
HORIZONS = {1: "15min", 2: "30min", 4: "60min"}

Mounted at /content/drive


(CSV FILE data load + 34 DAYS SPLITED INTO 24/5/5 DAYS)

In [2]:
data = {}
for h in HORIZONS:
    d = pd.read_csv(folder + f"df_clean_h{h}.csv", parse_dates=["DATE_TIME", "TARGET_TIME"])
    data[h] = d.sort_values("DATE_TIME").reset_index(drop=True)

# Define the split ONCE from the calendar days of the h=1 dataset
days = np.array(sorted(data[1]["DATE_TIME"].dt.normalize().unique()))
n_days = len(days)
n_train = int(round(n_days * 0.70))
n_val = int(round(n_days * 0.15))
train_days = days[:n_train]
val_days = days[n_train:n_train + n_val]
test_days = days[n_train + n_val:]

print(f"Total days: {n_days}")
print(f"Train: {len(train_days)} days ({pd.Timestamp(train_days[0]).date()} to {pd.Timestamp(train_days[-1]).date()})")
print(f"Val:   {len(val_days)} days ({pd.Timestamp(val_days[0]).date()} to {pd.Timestamp(val_days[-1]).date()})")
print(f"Test:  {len(test_days)} days ({pd.Timestamp(test_days[0]).date()} to {pd.Timestamp(test_days[-1]).date()})")

split_of_day = {}
for name, ds in (("train", train_days), ("val", val_days), ("test", test_days)):
    for dd in ds:
        split_of_day[pd.Timestamp(dd)] = name

Total days: 34
Train: 24 days (2020-05-15 to 2020-06-07)
Val:   5 days (2020-06-08 to 2020-06-12)
Test:  5 days (2020-06-13 to 2020-06-17)


split apply + purge + leakage check

In [4]:
for h, d in data.items():
    s_feat = d["DATE_TIME"].dt.normalize().map(split_of_day)
    s_targ = d["TARGET_TIME"].dt.normalize().map(split_of_day)
    keep = (s_feat == s_targ) & s_feat.notna()          # purge rows whose target crosses a split boundary
    out = d[keep].copy()
    out["SPLIT"] = s_feat[keep]
    data[h] = out.reset_index(drop=True)

    # ---- leakage assertions ----
    tr = out[out.SPLIT == "train"]; va = out[out.SPLIT == "val"]; te = out[out.SPLIT == "test"]
    assert tr["TARGET_TIME"].max() < va["DATE_TIME"].min(), f"h={h}: train targets reach into val"
    assert va["TARGET_TIME"].max() < te["DATE_TIME"].min(), f"h={h}: val targets reach into test"
    assert tr["DATE_TIME"].max() < va["DATE_TIME"].min() < va["DATE_TIME"].max() < te["DATE_TIME"].min()

    print(f"h={h} ({HORIZONS[h]}): train={len(tr)}  val={len(va)}  test={len(te)}  "
          f"(purged {len(d) - len(out)} boundary rows)")

print("\nNo temporal overlap, and no target crosses a split boundary, for any horizon.")

h=1 (15min): train=2159  val=479  test=473  (purged 0 boundary rows)
h=2 (30min): train=2152  val=478  test=471  (purged 0 boundary rows)
h=4 (60min): train=2139  val=476  test=469  (purged 0 boundary rows)

No temporal overlap, and no target crosses a split boundary, for any horizon.


### Weather character of each split (daily insolation)

Daily insolation = sum of IRRADIATION (kW/m²) × 0.25 h → kWh/m² per day.
Use this in the report to state how sunny/cloudy the test days were.

(daily insolation table)

In [5]:
d1 = data[1].copy()
d1["date"] = d1["DATE_TIME"].dt.normalize()
daily = d1.groupby("date").agg(split=("SPLIT", "first"),
                               insolation_kWh_m2=("IRRADIATION", lambda x: x.sum() * 0.25),
                               peak_irradiation=("IRRADIATION", "max"))
print(daily.round(2).to_string())
print()
print(daily.groupby("split")["insolation_kWh_m2"].agg(["mean", "min", "max"]).round(2))

            split  insolation_kWh_m2  peak_irradiation
date                                                  
2020-05-15  train               4.76              0.89
2020-05-16  train               4.66              0.81
2020-05-17  train               5.73              1.00
2020-05-18  train               3.82              0.97
2020-05-19  train               3.87              0.81
2020-05-20  train               4.26              0.98
2020-05-21  train               5.62              1.04
2020-05-22  train               5.53              1.05
2020-05-23  train               6.46              1.11
2020-05-24  train               6.23              0.98
2020-05-25  train               6.78              0.97
2020-05-26  train               6.06              1.05
2020-05-27  train               5.13              1.04
2020-05-28  train               6.51              1.06
2020-05-29  train               6.15              1.05
2020-05-30  train               5.47              1.15
2020-05-31

The 5-day insolation during the test (4.16 to 6.35) is similar to that of the train [dataset/conditions], meaning the weather during the test was neither excessively cloudy nor overly sunny.

In [6]:
for h, d in data.items():
    d.to_csv(folder + f"ds_h{h}_split.csv", index=False)
print("Saved ds_h1_split.csv, ds_h2_split.csv, ds_h4_split.csv (with a SPLIT column)")

Saved ds_h1_split.csv, ds_h2_split.csv, ds_h4_split.csv (with a SPLIT column)
